Install all dependencies

In [1]:
!pip install torch transformers ranx accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.0/318.0 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.3/228.3 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.8/1

Download the collections

In [2]:
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EaBHP-PszZhBr3f4dfAFA3MBAE_XTB6k-iW4mgUf5dYjbg?e=GFuQLT&download=1" -O pubmed_2022_tiny.jsonl.gz
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EXZIa3anvQ5DmTZ9MspBspMB6IIRORb1Wrb_a9lTb3GbIA?e=Ngdgai&download=1" -O pubmed_2022_small.jsonl.gz

--2023-12-13 11:43:46--  https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EaBHP-PszZhBr3f4dfAFA3MBAE_XTB6k-iW4mgUf5dYjbg?e=GFuQLT&download=1
Resolving uapt33090-my.sharepoint.com (uapt33090-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to uapt33090-my.sharepoint.com (uapt33090-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/joao_fonseca_ua_pt/Documents/UC/RI/P/A02/repository/data/pubmed_2022_tiny.jsonl.gz?ga=1 [following]
--2023-12-13 11:43:47--  https://uapt33090-my.sharepoint.com/personal/joao_fonseca_ua_pt/Documents/UC/RI/P/A02/repository/data/pubmed_2022_tiny.jsonl.gz?ga=1
Reusing existing connection to uapt33090-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 134376760 (128M) [application/x-gzip]
Saving to: ‘pubmed_2022_tiny.jsonl.gz’

pubmed_2022_tiny.js 100%[===================>] 128.15M  28.8MB/s    in 5.7s    

2023

Clone GitHub repository and copy data

In [3]:
!git clone https://github.com/joaompfonseca/ri-neural-reranker.git
!cp ri-neural-reranker/data/* .
!cp ri-neural-reranker/trainer/* .

Cloning into 'ri-neural-reranker'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 42 (delta 6), reused 34 (delta 2), pack-reused 0
Receiving objects: 100% (42/42), 20.25 MiB | 4.97 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Updating files: 100% (23/23), done.


Import all modules

In [4]:
import gzip
import json
import math
import torch

from collator import RankingCollator
from collections import defaultdict
from data import BioASQDataset, BioASQPointwiseIterator, InferenceRankingIterator, InferenceDataset
from google.colab import drive
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments
from ranker_trainer import RankerTrainer
from sampler import BasicSampler
from sklearn.metrics import precision_score, recall_score
from tqdm import tqdm
from utils import load_collection_lookup

/usr/local/lib/python3.10/dist-packages/transformers/deepspeed.py:23: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Train the model

Preview the train dataset

In [ ]:
import json

with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    print(data["query"])
    print(data["pos_docs"])
    print(data["neg_docs"])
    break

Choose the model checkpoint

In [ ]:
model_checkpoint = "bert-base-uncased"

Choose the device

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Configure the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2).to("cuda")

Configure the tokenizer

In [ ]:
TOKENIZER_LENGTH = 512

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.model_max_length = TOKENIZER_LENGTH

Prepare collection, dataset and relevant documents to feed into dataset

In [ ]:
MAX_QUERIES = 500 # out of 2471

In [ ]:
collection = dict()
dataset = dict()
qrels = dict()

# Collection - Tiny because train dataset negatives are taken from it
with gzip.open("pubmed_2022_tiny.jsonl.gz", "r") as all_file:
  for i, line in enumerate(all_file):
    print(f"Loading {i} documents from tiny dataset...", end="\r")
    doc = json.loads(line)
    collection[doc["pmid"]] = doc["title"] + " " + doc["abstract"]

# Dataset and relevant documents
with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    query = data["query"]
    for i in range(len(data["pos_docs"])):
      collection[data["pos_docs"][i]["id"]] = data["pos_docs"][i]["text"]
      collection[data["neg_docs"][i]["id"]] = data["neg_docs"][i]["text"]
    pos_docs = [doc["id"] for doc in data["pos_docs"]]
    neg_docs = [doc["id"] for doc in data["neg_docs"]]
    dataset[query] = {"question": query, "pos_docs": pos_docs, "neg_docs": neg_docs}
    qrels[query] = {docid: 1 for docid in pos_docs}

train_dataset = BioASQDataset(
  dataset=dataset,
  tokenizer=tokenizer,
  qrels_dict=qrels,
  collection=collection,
  iterator_class=BioASQPointwiseIterator[BasicSampler],
  max_questions=MAX_QUERIES
)

Configure the training arguments

In [ ]:
BATCH_SIZE       = 8
LEARNING_RATE    = 2e-5 # AdamW
NUMBER_OF_EPOCHS = 5

In [ ]:
training_args = TrainingArguments(
  num_train_epochs=NUMBER_OF_EPOCHS,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  per_device_train_batch_size=BATCH_SIZE,
  dataloader_pin_memory=True,
  output_dir="train_output",
  logging_strategy="steps",
  logging_first_step=True,
  logging_steps=100,
  save_strategy="epoch",
  save_total_limit=2,
  seed=42
)

trainer = RankerTrainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  tokenizer=tokenizer,
  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
  preprocess_logits_for_metrics=lambda logits, labels: torch.nn.functional.softmax(logits, dim=-1)[:,1],
)

Train the model!

In [ ]:
trainer.train()

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Token indices sequence length is longer than the specified maximum sequence length for this model (755 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss
1,0.703800
100,0.201300
200,0.027300
300,0.057700
400,0.009000
500,0.038700
600,0.014100
700,0.023400
800,0.017600
900,0.033000


TrainOutput(global_step=4760, training_loss=0.017651909840598452, metrics={'train_runtime': 3340.6882, 'train_samples_per_second': 11.396, 'train_steps_per_second': 1.425, 'total_flos': 8891281060453680.0, 'train_loss': 0.017651909840598452, 'epoch': 5.0})

Zip the train results and mount Drive to copy them there

In [ ]:
!zip -r train_output.zip train_output/
drive.mount('/content/drive')
!cp train_output.zip drive/train_output.zip

# Rerank BM25 using the model

Download the model

In [5]:
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/ERPQDz2GemFNr_2IUBa8mtYBN3URb6orjW73vHnLjxt87Q?e=HEdAeL&download=1" -O train_output.zip
!unzip train_output.zip

--2023-12-13 11:44:40--  https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/ERPQDz2GemFNr_2IUBa8mtYBN3URb6orjW73vHnLjxt87Q?e=HEdAeL&download=1
Resolving uapt33090-my.sharepoint.com (uapt33090-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to uapt33090-my.sharepoint.com (uapt33090-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/joao_fonseca_ua_pt/Documents/UC/RI/P/A02/repository/model/train_output.zip?ga=1 [following]
--2023-12-13 11:44:41--  https://uapt33090-my.sharepoint.com/personal/joao_fonseca_ua_pt/Documents/UC/RI/P/A02/repository/model/train_output.zip?ga=1
Reusing existing connection to uapt33090-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 2257756284 (2.1G) [application/x-zip-compressed]
Saving to: ‘train_output.zip’

train_output.zip    100%[===================>]   2.10G  69.9MB/s    in 35s     

2023-12-13 11:45:1

Choose the model checkpoint

In [6]:
model_checkpoint = "train_output/checkpoint-4760"

Choose the device

In [7]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Configure the model

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint).to(DEVICE)

Configure the tokenizer

In [9]:
TOKENIZER_LENGTH = 512

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.model_max_length = TOKENIZER_LENGTH

Load tiny collection for BM25 runs

In [16]:
RUN = "tiny" # small
COLLECTION = {"tiny": "pubmed_2022_tiny.jsonl.gz", "small": "pubmed_2022_small.jsonl.gz"}
BM25_RESULT = {"tiny": "BM25_E8B1.jsonl", "small": "BM25_E8B2.jsonl"}
BATCH_SIZE = 32

In [17]:
collection = dict()
query_id_to_query = dict()

with gzip.open(COLLECTION[RUN], "r") as all_file:
  for i, line in enumerate(all_file):
    print(f"Loading {i} documents from {RUN} dataset...", end="\r")
    doc = json.loads(line)
    collection[doc["pmid"]] = doc["title"] + " " + doc["abstract"]

with open(BM25_RESULT[RUN], "r") as bm25_results:
  for line in bm25_results:
    run = json.loads(line)
    query_id_to_query[run["id"]] = run["question"]

bm25_dataset = InferenceDataset(
  BM25_RESULT[RUN],
  collection,
  tokenizer,
  at=100,
  iterator_class=InferenceRankingIterator
)

dataloader = torch.utils.data.DataLoader(
  bm25_dataset,
  batch_size=BATCH_SIZE,
  pin_memory=True,
  collate_fn=RankingCollator(tokenizer)
)

Rerank BM_25 results using the model!

In [18]:
bm25_rerank = defaultdict(list)

for sample in tqdm(dataloader):
  _inputs = sample["inputs"].to(DEVICE)

  with torch.no_grad():
    logits = model(**_inputs).logits
    score = torch.nn.functional.softmax(logits, dim=-1)[:,1] # [0-1]

  for i, q_id in enumerate(sample["id"]):
    bm25_rerank[q_id].append({"id":sample["doc_id"][i],
                           "score":score[i].item()})

# Sort documents by relevance
for q_id in bm25_rerank:
  bm25_rerank[q_id].sort(key=lambda x:-x["score"])

100%|██████████| 313/313 [05:38<00:00,  1.08s/it]


Save results to file

In [20]:
with open(f"BM25_{RUN}_rerank.jsonl", "w") as out_file:
  for query_id, documents in bm25_rerank.items():
    entry = {"id": query_id, "question": query_id_to_query[query_id], "documents":documents}
    out_file.write(json.dumps(entry) + "\n")